In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'CMacedonia'

In [4]:
YEAR = 2023
MONTH = 'June'
PERIOD = '1st'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model_new.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler_new.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer_new.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.46197,40.60446,2023-06-01,κεντρικης μακεδονιας,αλεξανδρειας,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,478.8,38293.0,86.8,46.40326,0.332313,0.019755,-0.316834,-0.019755,0.322489,0.013944,-0.310816,-0.013944,0.072215,0.061938,0.052239,0.061938,25.298420,33.776364,16.820476,11.510466,1.708510,11.778876,0.439805,21.186067,5.575849,24.421964,8.937268,3.167303,7.402818,117.491622,18483.883785,1629.598106,4,120.472193,7.866005,180.191380,0.0,1.400071,31,99,36,99.0,30,99,12,12,1,6,7,2,0,146,0,0
1,22.07722,41.01522,2023-06-01,κεντρικης μακεδονιας,αλμωπιας,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,985.8,24924.0,28.0,46.57952,0.586278,0.210013,-0.483424,-0.210013,0.537723,0.258753,-0.430061,-0.258753,0.063740,0.040410,0.052581,0.040410,17.250830,22.884231,11.617429,9.238694,1.920162,9.573419,-0.371554,15.120878,3.834306,16.704092,6.474975,6.353493,28.353780,206.167793,25701.766035,44.089931,2,194.572797,144.613300,181.767317,0.0,113.550432,31,93,36,94.0,30,93,12,12,3,5,8,2,0,129,0,0
2,22.89198,40.65603,2023-06-01,κεντρικης μακεδονιας,αμπελοκηπων μενεμενης,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,9.8,49674.0,5319.1,46.65786,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.010000,NaN,12.381429,3.596667,11.870000,2.525385,18.760769,5.121429,23.898000,11.870000,54.540328,73.751313,251.891959,1013.723139,1259.926671,2,166.963790,4.655965,180.457193,0.0,24.343007,32,97,9,99.0,30,97,13,13,10,8,9,2,0,148,0,0
3,23.95900,40.91196,2023-06-01,κεντρικης μακεδονιας,αμφιπολης,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,411.8,7169.0,22.3,47.41120,0.519238,0.248605,-0.431427,-0.248605,0.479682,0.239302,-0.398712,-0.239302,0.037599,0.041656,0.024496,0.041656,21.081744,26.947333,15.216154,10.655161,3.113550,9.895287,0.416503,16.682835,4.861432,20.539981,7.233860,1.724839,10.878010,112.399961,14413.246350,196.335052,5,220.810138,250.710581,185.823044,0.0,25.464784,31,88,30,88.0,30,88,10,10,1,6,6,2,0,78,0,0
4,23.69747,40.49593,2023-06-01,κεντρικης μακεδονιας,αριστοτελη,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,747.0,16994.0,24.5,46.92004,0.652838,0.342678,-0.520211,-0.342678,0.642374,0.356262,-0.508083,-0.356262,0.048104,0.042566,0.044359,0.042566,18.992768,25.312778,12.672759,10.968774,3.770478,9.453781,2.661346,15.362634,4.844974,17.259672,7.381085,0.642929,23.735742,236.203195,11468.297401,1749.385768,22,273.587292,768.043224,181.562024,0.0,1.033661,14,99,10,99.0,10,99,4,4,6,4,4,2,0,71,0,0


In [7]:
features_to_remove = ['x', 'y', 'eq_distance','day', 'month', 'week', 'year', 'lc_prop1_assessment',
                    'lc_prop2', 'lc_prop2_assessment', 'lc_prop3', 'lc_prop3_assessment',
                    'lc_type2', 'lc_type3', 'lc_type4', 'lc_type5', 'lw', 'qc','ndvi_mean', 'ndmi_mean', 'ndwi_mean', 'ndbi_mean', 'ndvi_std',
                    'ndmi_std', 'ndwi_std', 'ndbi_std',]

In [8]:
object_cols = data_test.select_dtypes(include=['object']).columns.to_list()
removed_cols = features_to_remove

X_test = data_test.drop(columns = ['case'] + object_cols + removed_cols)
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [9]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.95369,40.62334,θεσσαλονικης,1,6,2023,0.035654
1,22.37828,40.28551,κατερινης,1,6,2023,0.013057
2,22.19377,40.47808,βεροιας,1,6,2023,0.011917
3,23.08410,40.49006,θερμης,1,6,2023,0.011461
4,23.67524,41.04497,εμμανουηλ παππα,1,6,2023,0.010037
5,22.36451,40.79600,πελλας,1,6,2023,0.008880
6,22.90397,41.04735,κιλκις,1,6,2023,0.008519
7,22.91743,40.42897,θερμαϊκου,1,6,2023,0.007529
8,23.57073,40.89405,βισαλτιας,1,6,2023,0.007164
9,22.90326,40.67609,κορδελιου ευοσμου,1,6,2023,0.007111


In [10]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0318551970475366, 0.3213778408024378, 0.9413989549667096, 0.9750382927661556, 0.9844280637528458, 1.0]


In [11]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.95369,40.62334,θεσσαλονικης,1,6,2023,0.035654,1
1,22.37828,40.28551,κατερινης,1,6,2023,0.013057,0
2,22.19377,40.47808,βεροιας,1,6,2023,0.011917,0
3,23.08410,40.49006,θερμης,1,6,2023,0.011461,0
4,23.67524,41.04497,εμμανουηλ παππα,1,6,2023,0.010037,0
5,22.36451,40.79600,πελλας,1,6,2023,0.008880,0
6,22.90397,41.04735,κιλκις,1,6,2023,0.008519,0
7,22.91743,40.42897,θερμαϊκου,1,6,2023,0.007529,0
8,23.57073,40.89405,βισαλτιας,1,6,2023,0.007164,0
9,22.90326,40.67609,κορδελιου ευοσμου,1,6,2023,0.007111,0


In [12]:
# results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}.csv", encoding = enc, index = False)

In [13]:
##TODO Visualisation of results